# Module 7: Transfer Learning and Fine-Tuning Concepts

## Learning Objectives
By the end of this module, you will be able to:
- Understand what transfer learning is and why it's powerful
- Recognize when to use full fine-tuning vs. other approaches
- Understand the risks of catastrophic forgetting
- Choose the right adaptation strategy for your use case

---

## 1. What is Transfer Learning?

**Transfer learning** is the practice of using a model trained on one task as a starting point for a different (but related) task.

### The Core Idea

```
Traditional ML:
  Task A → Train from scratch → Model A
  Task B → Train from scratch → Model B

Transfer Learning:
  Task A (large data) → Train from scratch → Pre-trained Model
                                                    ↓
  Task B (small data) ← Fine-tune ←────────────────┘
```

### Why It Works

Neural networks learn **hierarchical representations**:

| Layer Type | Image Models | Language Models |
|------------|--------------|------------------|
| **Early layers** | Edges, textures | Characters, common words |
| **Middle layers** | Shapes, patterns | Phrases, syntax |
| **Late layers** | Objects, scenes | Semantics, context |

These lower-level features are **transferable** across many tasks!

### Real-World Impact

```
ImageNet Pre-training (Computer Vision)
├── Medical imaging classification
├── Satellite image analysis
├── Manufacturing defect detection
└── Self-driving car perception

LLM Pre-training (NLP)
├── Sentiment analysis
├── Named entity recognition
├── Question answering
├── Code generation
└── Summarization
```

**Key Benefit**: Turn weeks of training into hours (or minutes) of fine-tuning!

---

## 2. Pre-trained Models in NLP

### The Pre-training Revolution

Before 2018: Train task-specific models from scratch

After 2018 (BERT, GPT): Pre-train on massive corpora, fine-tune for specific tasks

### How LLMs Are Pre-trained

| Model | Pre-training Task | Training Data |
|-------|-------------------|---------------|
| **BERT** | Masked word prediction + Next sentence | Wikipedia, Books |
| **GPT** | Next word prediction | Web text |
| **T5** | Text-to-text (various tasks) | C4 corpus |
| **LLaMA** | Next word prediction | Various sources |

### Training Scale

```
Model Size        Training Cost       Your Fine-tuning
───────────────────────────────────────────────────────
GPT-2 (1.5B)      ~$50,000           ~$100
GPT-3 (175B)      ~$4,600,000        ~$500
LLaMA-2 (70B)     ~$2,000,000        ~$1,000
```

Pre-training is expensive; fine-tuning is cheap!

In [ ]:
!pip install -q transformers torch

In [ ]:
# Example: Loading a pre-trained model
from transformers import AutoModel, AutoTokenizer

# Load pre-trained BERT (this downloads the weights trained on massive data)
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

print(f"✅ Loaded {model_name}")
print(f"   Parameters: {model.num_parameters():,}")
print(f"   These weights encode knowledge from billions of words!")

---

## 3. Fine-Tuning Strategies

### Full Fine-Tuning

Update **all** model parameters on your task-specific data.

```
Pre-trained Model
      ↓
[All layers unfrozen]
      ↓
Train on your data
      ↓
Fine-tuned Model
```

**Pros:**
- Maximum adaptation to your task
- Best performance potential

**Cons:**
- Computationally expensive
- Risk of catastrophic forgetting
- Needs more training data

### Feature Extraction (Frozen)

Freeze all pre-trained layers, only train a new task head.

```
Pre-trained Model (frozen)
      ↓
[New task head (trainable)]
      ↓
Train only the head
```

**Pros:**
- Fast training
- Preserves pre-trained knowledge
- Works with small datasets

**Cons:**
- Limited adaptation
- May underperform on very different tasks

### Partial Fine-Tuning

Freeze early layers, fine-tune later layers + task head.

```
[Early layers - frozen]     (general features)
         ↓
[Later layers - trainable]  (task-specific features)
         ↓
[Task head - trainable]
```

**Good balance** between adaptation and efficiency.

### Modern Efficient Methods

| Method | Trainable Params | Idea |
|--------|------------------|------|
| **LoRA** | ~0.1% | Low-rank adapters in attention layers |
| **Prefix-tuning** | ~0.1% | Learnable prefix tokens |
| **Prompt-tuning** | <0.01% | Learnable soft prompts |
| **Adapters** | ~1% | Small trainable modules between layers |

These are becoming the standard for LLM adaptation!

In [ ]:
# Visualize layer freezing strategies
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 4, figsize=(16, 6))

strategies = [
    ("Full Fine-Tuning", [1, 1, 1, 1, 1, 1]),
    ("Feature Extraction", [0, 0, 0, 0, 0, 1]),
    ("Partial Fine-Tuning", [0, 0, 0, 1, 1, 1]),
    ("LoRA/Adapters", [0.3, 0.3, 0.3, 0.3, 0.3, 1]),
]

layer_names = ['Embed', 'Layer 1', 'Layer 2', 'Layer 3', 'Layer N', 'Head']

for ax, (name, trainable) in zip(axes, strategies):
    colors = ['coral' if t == 1 else 'steelblue' if t == 0 else 'plum' for t in trainable]
    
    for i, (layer, color) in enumerate(zip(layer_names, colors)):
        rect = mpatches.FancyBboxPatch((0.2, 0.8 - i*0.13), 0.6, 0.1,
                                       boxstyle="round,pad=0.01",
                                       facecolor=color, edgecolor='black')
        ax.add_patch(rect)
        ax.text(0.5, 0.85 - i*0.13, layer, ha='center', va='center', fontsize=9)
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_title(name, fontsize=11, fontweight='bold')
    ax.axis('off')

# Legend
frozen_patch = mpatches.Patch(color='steelblue', label='Frozen')
trainable_patch = mpatches.Patch(color='coral', label='Trainable')
partial_patch = mpatches.Patch(color='plum', label='Partially Trainable')
fig.legend(handles=[frozen_patch, trainable_patch, partial_patch], 
           loc='lower center', ncol=3, fontsize=10)

plt.suptitle('Fine-Tuning Strategies', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.show()

---

## 4. Catastrophic Forgetting

### The Problem

When you fine-tune a model on new data, it can **forget** what it learned during pre-training.

```
Pre-trained Model: Knows grammar, facts, reasoning
                           ↓
                   Fine-tune on medical data
                           ↓
Fine-tuned Model: Great at medical tasks!
                  BUT... forgot general grammar? 😱
```

### Why It Happens

- Neural networks have **limited capacity**
- New gradients overwrite previous weights
- Fine-tuning data doesn't cover all original capabilities

### Mitigation Strategies

| Strategy | How It Helps |
|----------|-------------|
| **Lower learning rate** | Smaller updates, gentler adaptation |
| **Freeze early layers** | Preserve general features |
| **Regularization** | Penalize large weight changes |
| **Elastic Weight Consolidation** | Protect important weights |
| **LoRA/Adapters** | Don't modify original weights at all |
| **Data mixing** | Include some original training data |

In [ ]:
# Demonstration: Effect of learning rate on forgetting
import numpy as np

# Simulated performance metrics
epochs = np.arange(1, 11)

# High learning rate: task performance up, general performance down
high_lr_task = 1 - np.exp(-epochs * 0.5)
high_lr_general = 0.9 * np.exp(-epochs * 0.2)

# Low learning rate: slower adaptation, preserved general knowledge
low_lr_task = 1 - np.exp(-epochs * 0.2)
low_lr_general = 0.9 * np.exp(-epochs * 0.05)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, high_lr_task, 'g-', linewidth=2, label='Task Performance')
axes[0].plot(epochs, high_lr_general, 'r--', linewidth=2, label='General Knowledge')
axes[0].set_title('High Learning Rate (0.001)', fontsize=12)
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Performance')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 1)

axes[1].plot(epochs, low_lr_task, 'g-', linewidth=2, label='Task Performance')
axes[1].plot(epochs, low_lr_general, 'r--', linewidth=2, label='General Knowledge')
axes[1].set_title('Low Learning Rate (0.0001)', fontsize=12)
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Performance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

plt.suptitle('Catastrophic Forgetting: Learning Rate Effect', fontsize=14)
plt.tight_layout()
plt.show()

print("💡 Lower learning rates help preserve general knowledge!")

---

## 5. Choosing the Right Approach

### Decision Tree

```
Have a pre-trained model for your domain?
│
├── YES → How much task-specific data?
│         │
│         ├── Small (<1K examples) → Feature extraction or prompt-tuning
│         ├── Medium (1K-10K) → LoRA or partial fine-tuning
│         └── Large (>10K) → Full fine-tuning possible
│
└── NO → Is there a related pre-trained model?
         │
         ├── YES → Use it! Some transfer is better than none
         └── NO → Train from scratch (expensive)
```

### Cost-Performance Trade-offs

| Method | Data Needed | Compute | Performance |
|--------|-------------|---------|-------------|
| Prompt engineering | 0 | Low | Medium |
| Prompt-tuning | 100s | Low | Medium-High |
| LoRA | 1,000s | Medium | High |
| Full fine-tuning | 10,000s | High | Highest |

---

## 6. Practical Considerations

### Data Quality > Quantity

For fine-tuning:
- **Clean, curated data** beats raw volume
- **Diverse examples** prevent overfitting
- **Consistent format** helps learning
- **Representative edge cases** improve robustness

### Compute Requirements

| Model Size | Full Fine-tune | LoRA | Memory Required |
|------------|----------------|------|----------------|
| 125M (Small) | ✅ Easy | ✅ Easy | ~4GB |
| 1-3B (Medium) | ⚠️ Needs GPU | ✅ Easy | ~12-24GB |
| 7B (Large) | ❌ Expensive | ✅ Possible | ~28GB |
| 70B+ (XL) | ❌ Very expensive | ⚠️ Tricky | 100GB+ |

### Best Practices

1. **Start small**: Try prompt engineering first
2. **Validate baseline**: Measure pre-trained performance
3. **Use validation set**: Monitor for overfitting
4. **Save checkpoints**: Enable rollback if needed
5. **Test original capabilities**: Check for forgetting

---

## 📝 Knowledge Check

### Question 1
A healthcare company wants to adapt GPT-4 for medical QA. They have 500 curated examples. What approach would you recommend?

<details>
<summary>Click for answer</summary>
<br>
✅ <b>Recommended: Prompt-tuning or few-shot prompting</b>
<br><br>
With only 500 examples, full fine-tuning risks overfitting. Better options:
<ul>
<li>Start with well-crafted prompts (zero cost)</li>
<li>Try few-shot prompting with examples in context</li>
<li>Consider OpenAI's fine-tuning API with their smaller models</li>
</ul>
</details>

### Question 2
What is catastrophic forgetting and how can LoRA help prevent it?

<details>
<summary>Click for answer</summary>
<br>
<b>Catastrophic Forgetting:</b> When fine-tuning overwrites weights encoding general knowledge, causing the model to lose abilities not covered by the fine-tuning data.
<br><br>
<b>How LoRA Helps:</b> LoRA doesn't modify the original weights! It adds small "adapter" matrices that learn task-specific adjustments. The original knowledge is preserved untouched.
</details>

### Question 3
Why do we often freeze early layers during partial fine-tuning?

<details>
<summary>Click for answer</summary>
<br>
Early layers learn <b>general, transferable features</b> (edges, basic patterns, common word patterns). These are:
<ul>
<li>Already well-trained on massive data</li>
<li>Useful across many tasks</li>
<li>Don't need task-specific adaptation</li>
</ul>
Later layers learn <b>task-specific features</b>, so they benefit more from fine-tuning.
</details>

---

## 🎯 Key Takeaways

1. **Transfer learning** lets you leverage expensive pre-training investments

2. **Full fine-tuning** gives maximum adaptation but is expensive and risky

3. **Catastrophic forgetting** is real - use lower learning rates and efficient methods

4. **LoRA, adapters, prompt-tuning** are modern alternatives that preserve original capabilities

5. **Start simple**: Prompt engineering → Few-shot → Fine-tuning

---

### Continue with:
- **02_fine_tuning_openai.ipynb** - Hands-on OpenAI fine-tuning
- **03_sampling_techniques.ipynb** - Control generation with temperature, top-p, etc.